# Module 7 — Pitcher Archetypes

**Classification order:** score all qualifying types, highest score wins.

| Type | Distinguishing signal |
|------|----------------------|
| P1 Power Ace | High velo + misses bats |
| P2 Craft Strikeout | Misses bats via deception (low velo ceiling) |
| P3 GB Craftsman | Weak contact, low hard hit allowed |
| P4 Stuff to Contact | Hard stuff → GB, allows more hard contact than P3 |

In [ ]:
import sys
sys.path.insert(0, "../modules")
from pitcher_archetypes import (
    classify_starter, build_starter_profile, build_bullpen_profile,
    starters_to_dataframe, compute_command_profile, compute_arsenal_depth,
)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Canonical test cases
archetype_metrics = {
    "deGrom (P1)": dict(avg_velo_pct=75,K_pct_pct=95,SwStr_pct_pct=92,stuff_composite_pct=88,CSW_pct_pct=90,HardHit_inv_pct=85,SpinEfficiency_pct=80,TunnelScore_pct=78,SwStr_per_velo_pct=70,FirstPitchStrike_pct=65,GB_pct_pct=50,HardHit_allowed_pct=15,BB_pct_pct=20,BB_pitch_inv_pct=80,Zone_pct_pct=70,Barrel_allowed_inv_pct=82,K_inv_pct=5),
    "Greinke (P2)": dict(avg_velo_pct=48,K_pct_pct=72,SwStr_pct_pct=68,SpinEfficiency_pct=85,TunnelScore_pct=82,SwStr_per_velo_pct=90,FirstPitchStrike_pct=80,stuff_composite_pct=52,CSW_pct_pct=72,HardHit_inv_pct=70,GB_pct_pct=55,HardHit_allowed_pct=30,BB_pct_pct=18,BB_pitch_inv_pct=82,Zone_pct_pct=75,Barrel_allowed_inv_pct=75,K_inv_pct=28),
    "Hendricks (P3)": dict(avg_velo_pct=25,K_pct_pct=45,SwStr_pct_pct=40,GB_pct_pct=72,HardHit_allowed_pct=20,HardHit_inv_pct=80,Barrel_allowed_inv_pct=78,BB_pct_pct=22,BB_pitch_inv_pct=78,K_inv_pct=55,Zone_pct_pct=68,SpinEfficiency_pct=60,TunnelScore_pct=65,SwStr_per_velo_pct=55,FirstPitchStrike_pct=70,stuff_composite_pct=30,CSW_pct_pct=50),
    "Sinker-P4":     dict(avg_velo_pct=68,K_pct_pct=42,SwStr_pct_pct=35,GB_pct_pct=65,HardHit_allowed_pct=58,HardHit_inv_pct=42,Barrel_allowed_inv_pct=55,BB_pct_pct=45,BB_pitch_inv_pct=55,K_inv_pct=58,Zone_pct_pct=50,stuff_composite_pct=60,CSW_pct_pct=42,SpinEfficiency_pct=50,TunnelScore_pct=48,SwStr_per_velo_pct=38,FirstPitchStrike_pct=55),
}
for name, m in archetype_metrics.items():
    r = classify_starter(m)
    print(f"{name:18s}: {r["type_code"]} {r["type"]:22s} score={r["score"]:.1f} conf={r["display_confidence"]}%")

In [ ]:
# Score bar chart
import numpy as np
labels = list(archetype_metrics.keys())
scores = [classify_starter(m)["score"] for m in archetype_metrics.values()]
codes  = [classify_starter(m)["type_code"] for m in archetype_metrics.values()]
colors = {"P1":"#e74c3c","P2":"#3498db","P3":"#2ecc71","P4":"#f39c12"}
fig,ax = plt.subplots(figsize=(9,4))
ax.barh(labels, scores, color=[colors.get(c,"grey") for c in codes])
ax.set_xlim(0,100)
ax.axvline(50,color="grey",linestyle="--",alpha=0.4)
ax.set_xlabel("Archetype Score")
ax.set_title("Pitcher Archetype Scores")
for i,(s,code) in enumerate(zip(scores,codes)):
    if s: ax.text(s+1,i,f"{code} {s:.0f}",va="center",fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Bullpen collective profile example
bp_metrics = dict(velocity_pct=70,attack_pct=55,HardHit_inv_pct=72,Barrel_inv_pct=68,K_pct_pct=80,GB_pct_pct=35,LevWPA_conc_pct=75,Platoon_bal_pct=60)
bp = build_bullpen_profile("NYY",2023,bp_metrics)
print(f"out_mechanism:      {bp["out_mechanism"]}")
print(f"leverage_structure: {bp["leverage_structure"]}")
for dim,score in bp["scores"].items():
    print(f"  {dim:22s}: {score:.1f}" if score else f"  {dim}: N/A")